# Week 2, Day 2: Social Media Analytics - Raw Dataset Generation

This notebook builds a **raw, intentionally messy** social media dataset (1000 rows, 20 columns).
Cleaning and metric calculation come in later notebooks, so nothing is fixed here.

| Layer | Columns |
|---|---|
| Identity | `post_id`, `account_id`, `platform`, `content_format`, `post_datetime` |
| Audience | `followers_at_post` |
| Exposure | `impressions`, `reach`, `views` |
| Engagement | `likes`, `comments`, `shares`, `saves` |
| Video | `video_length_sec`, `avg_watch_time_sec` |
| Conversion | `link_clicks`, `conversions`, `new_followers` |
| Content | `hashtag_count`, `caption_length` |

**Output:** `social_media_raw.csv` (same folder as this notebook). The random seed is fixed, so every run gives the same file.

## 1. Setup

In [1]:
import logging

logging.basicConfig(
    filename="w2d2.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger=logging.getLogger("w2d2")
print("Logging is set up. Check w2d2.log for log messages.")

Logging is set up. Check w2d2.log for log messages.


In [ ]:
logger.info("Week 2 Day 2: Logging is set up successfully.")

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

N_UNIQUE = 985      # unique posts
N_DUPES = 15        # duplicate rows appended -> 1000 rows total

## 2. Accounts and base posts

Eight accounts, two per platform. Each post is assigned to a random account, and the content format depends on the platform (TikTok and YouTube are mostly video).

In [3]:
accounts = pd.DataFrame({
    "account_id": [f"ACC{i:02d}" for i in range(1, 9)],
    "platform":   ["Instagram", "Instagram", "TikTok", "TikTok",
                   "YouTube", "YouTube", "Facebook", "Facebook"],
    "base_followers": [15000, 48000, 22000, 95000, 31000, 12000, 60000, 8000],
})

idx = rng.integers(0, len(accounts), N_UNIQUE)
df = accounts.iloc[idx].reset_index(drop=True)
df.insert(0, "post_id", [f"POST{i:04d}" for i in range(1, N_UNIQUE + 1)])

def pick_format(platform):
    if platform in ("TikTok", "YouTube"):
        return rng.choice(["video", "image", "text"], p=[0.85, 0.10, 0.05])
    return rng.choice(["video", "image", "carousel", "text"], p=[0.35, 0.35, 0.20, 0.10])

df["content_format"] = df["platform"].apply(pick_format)
df.head()

,post_id,account_id,platform,base_followers,content_format
0,POST0001,ACC01,Instagram,15000,video
1,POST0002,ACC07,Facebook,60000,carousel
2,POST0003,ACC06,YouTube,12000,video
3,POST0004,ACC04,TikTok,95000,video
4,POST0005,ACC04,TikTok,95000,video


## 3. Post datetime and follower count

In [ ]:
start = pd.Timestamp("2026-01-01")
days = rng.integers(0, 181, N_UNIQUE)
hour_weights = np.array([1, 1, 1, 1, 1, 2, 3, 4, 5, 5, 5, 6,
                         6, 5, 5, 5, 6, 7, 8, 9, 8, 6, 4, 2])
hours = rng.choice(range(24), N_UNIQUE, p=hour_weights / hour_weights.sum())
minutes = rng.integers(0, 60, N_UNIQUE)

df["post_datetime"] = (start + pd.to_timedelta(days, unit="D")
                       + pd.to_timedelta(hours, unit="h")
                       + pd.to_timedelta(minutes, unit="m"))

# followers grow ~0.15% per day from the base
df["followers_at_post"] = (df["base_followers"] * (1 + 0.0015 * days)
                           * rng.normal(1, 0.01, N_UNIQUE)).round().astype(int)
df = df.drop(columns="base_followers")
df[["post_id", "post_datetime", "followers_at_post"]].head()

## 4. Exposure metrics

Reach depends on format and follower count. Impressions = reach x frequency (always at least 1). Views only exist for video posts.

In [ ]:
fmt_reach = {"video": 0.55, "image": 0.35, "carousel": 0.40, "text": 0.20}
reach_pct = df["content_format"].map(fmt_reach) * rng.lognormal(0, 0.45, N_UNIQUE)
df["reach"] = (df["followers_at_post"] * reach_pct).round().astype(int)

frequency = 1 + rng.exponential(0.35, N_UNIQUE)
df["impressions"] = (df["reach"] * frequency).round().astype(int)

is_video = df["content_format"].eq("video")
view_share = rng.uniform(0.55, 0.90, N_UNIQUE)
df["views"] = np.where(is_video, (df["impressions"] * view_share).round(), 0).astype(int)
df[["reach", "impressions", "views"]].describe().round(0)

## 5. Engagement metrics

In [ ]:
fmt_eng = {"video": 1.30, "image": 0.90, "carousel": 1.10, "text": 0.50}
eng_mult = df["content_format"].map(fmt_eng)

def counts(rate_low, rate_high):
    rate = rng.uniform(rate_low, rate_high, N_UNIQUE) * eng_mult
    return rng.poisson(df["reach"] * rate)

df["likes"]    = counts(0.030, 0.070)
df["comments"] = counts(0.003, 0.010)
df["shares"]   = counts(0.001, 0.008)
df["saves"]    = counts(0.002, 0.012)
df[["likes", "comments", "shares", "saves"]].describe().round(0)

## 6. Video watch, conversion and content attributes

In [ ]:
# video watch (video posts only)
df["video_length_sec"] = np.where(is_video, rng.choice([15, 30, 45, 60, 90, 180, 300], N_UNIQUE), np.nan)
watch_frac = rng.beta(2.5, 3.0, N_UNIQUE)
df["avg_watch_time_sec"] = np.where(is_video, (df["video_length_sec"] * watch_frac).round(1), np.nan)

# conversion
df["link_clicks"] = rng.poisson(df["impressions"] * rng.uniform(0.004, 0.015, N_UNIQUE))
df["conversions"] = rng.binomial(df["link_clicks"], rng.uniform(0.02, 0.08, N_UNIQUE))
df["new_followers"] = rng.poisson(df["reach"] * rng.uniform(0.0005, 0.004, N_UNIQUE))

# content attributes
df["hashtag_count"] = rng.integers(0, 16, N_UNIQUE)
df["caption_length"] = rng.integers(0, 400, N_UNIQUE)
df.shape

## 7. Injecting raw-data defects

Everything below is deliberate. The defects will be found and fixed in the cleaning notebook.

| # | Defect | Cleaning skill it practices |
|---|---|---|
| 1 | Missing values | Imputation vs. dropping |
| 2 | Inflated likes (outliers) | Outlier detection |
| 3 | Reach greater than impressions | Logical validation |
| 4 | Negative shares | Range checks |
| 5 | Zero followers | Range checks |
| 6 | Inconsistent labels | String normalization |
| 7 | Mixed datetime formats | Date parsing |
| 8 | Numbers stored as text | Type conversion |
| 9 | Duplicate rows | De-duplication |

In [ ]:
# 1. Missing values (blank cells)
for col, frac in {"impressions": 0.03, "reach": 0.02, "likes": 0.03, "comments": 0.03,
                  "shares": 0.03, "saves": 0.04, "link_clicks": 0.04,
                  "followers_at_post": 0.02, "avg_watch_time_sec": 0.05,
                  "hashtag_count": 0.03, "caption_length": 0.03}.items():
    df[col] = df[col].astype("float")
    df.loc[rng.random(N_UNIQUE) < frac, col] = np.nan

# 2. Outliers: a few posts with wildly inflated likes
mask = rng.random(N_UNIQUE) < 0.005
df.loc[mask, "likes"] = df.loc[mask, "likes"] * 100

# 3. Logical errors: reach greater than impressions
mask = rng.random(N_UNIQUE) < 0.010
df.loc[mask, "reach"] = df.loc[mask, "impressions"] * 1.4

# 4. Negative values
mask = rng.random(N_UNIQUE) < 0.005
df.loc[mask, "shares"] = -df.loc[mask, "shares"].abs()

# 5. Zero followers
mask = rng.random(N_UNIQUE) < 0.005
df.loc[mask, "followers_at_post"] = 0

In [ ]:
# 6. Inconsistent labels (case, spaces, aliases)
plat_variants = {"Instagram": ["instagram", "INSTAGRAM", "Insta", " Instagram "],
                 "TikTok":    ["tiktok", "Tik Tok", "TIKTOK", "TikTok "],
                 "YouTube":   ["youtube", "Youtube", "YT", "YOUTUBE"],
                 "Facebook":  ["facebook", "FB", "FACEBOOK", "Facebook "]}
mask = rng.random(N_UNIQUE) < 0.15
df.loc[mask, "platform"] = df.loc[mask, "platform"].apply(lambda p: rng.choice(plat_variants[p]))

mask = rng.random(N_UNIQUE) < 0.12
df.loc[mask, "content_format"] = df.loc[mask, "content_format"].apply(
    lambda f: rng.choice([f.upper(), f.capitalize(), f + " ", "Reel" if f == "video" else f]))

# 7. Mixed datetime formats (stored as text)
def fmt_dt(ts):
    r = rng.random()
    if r < 0.60:
        return ts.strftime("%Y-%m-%d %H:%M:%S")
    if r < 0.85:
        return ts.strftime("%d/%m/%Y %H:%M")
    return ts.strftime("%b %d, %Y %I:%M %p")

df["post_datetime"] = df["post_datetime"].apply(fmt_dt)

# 8. Numbers stored as text ("1.2K", "N/A", "-")
for col in ["likes", "link_clicks"]:
    df[col] = df[col].astype(object)

mask = rng.random(N_UNIQUE) < 0.010
df.loc[mask, "likes"] = df.loc[mask, "likes"].apply(
    lambda v: f"{v / 1000:.1f}K" if pd.notna(v) and v >= 1000 else v)
mask = rng.random(N_UNIQUE) < 0.010
df.loc[mask, "link_clicks"] = rng.choice(["N/A", "-", "n/a"], mask.sum())

## 8. Duplicates, shuffle and save

In [ ]:
# 9. Duplicate rows, then shuffle so duplicates are not adjacent
dupes = df.sample(N_DUPES, random_state=1)
df = pd.concat([df, dupes], ignore_index=True).sample(frac=1, random_state=7).reset_index(drop=True)

cols = ["post_id", "account_id", "platform", "content_format", "post_datetime",
        "followers_at_post", "impressions", "reach", "views", "likes", "comments",
        "shares", "saves", "video_length_sec", "avg_watch_time_sec",
        "link_clicks", "conversions", "new_followers", "hashtag_count", "caption_length"]
df = df[cols]
df.to_csv("social_media_raw.csv", index=False)
print("Saved social_media_raw.csv | shape:", df.shape)

## 9. Quick look at the raw file

In [ ]:
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
raw = pd.read_csv("social_media_raw.csv")
raw.head(8)

In [ ]:
print("Shape:", raw.shape)
print("\nDuplicate rows:", raw.duplicated().sum())
print("\nMissing values per column:")
print(raw.isna().sum())

In [ ]:
print("Data types as loaded:")
print(raw.dtypes)
print("\nPlatform labels (20 spellings for 4 platforms):")
print(raw["platform"].value_counts())

## Next steps (cleaning notebook)

1. Profile the raw data in detail
2. Remove duplicates
3. Standardize `platform` and `content_format`
4. Parse `post_datetime`, extract weekday and hour
5. Convert `likes` and `link_clicks` to numeric
6. Validate logic (reach <= impressions, no negatives, followers > 0)
7. Handle missing values and outliers
8. Compute all social media metrics